# Part 3: Adversarial Attacks - Breaking the Classifier

This notebook implements two attacks from scratch:
1. Character-level evasion attack (zero-width spaces + homoglyphs + random duplication)
2. Label-flipping poisoning attack (5% flipped labels)

It reports:
- Attack 1: Attack Success Rate (ASR), sample size, and average confidence before/after perturbation
- Attack 2: clean vs poisoned model comparison table (Accuracy, F1, FNR) on the same clean evaluation subset

In [1]:
# Uncomment if needed in your environment.
# !pip install -q transformers datasets accelerate scikit-learn seaborn pandas matplotlib torch

import os
import re
import random
import warnings

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments, set_seed

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

d:\Fast\semester_8\responsible_ai\assignment_02\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


In [2]:
DATA_PATH = os.path.join("dataset", "jigsaw-unintended-bias-train.csv")
USECOLS = ["comment_text", "toxic", "black", "white"]

PART1_CHECKPOINT_DIR = os.path.join("checkpoints", "part1_distilbert_baseline")
MODEL_BASE = "distilbert-base-uncased"

ATTACK1_THRESHOLD_FOR_TOXIC_LABEL = 0.5
ATTACK1_CONFIDENCE_MIN = 0.7
ATTACK1_SAMPLE_SIZE = 500

POISON_FRAC = 0.05
POISON_EPOCHS = 3

if not os.path.isdir(PART1_CHECKPOINT_DIR):
    raise FileNotFoundError(
        f"Missing Part 1 checkpoint at {PART1_CHECKPOINT_DIR}. Run part1.ipynb first so Part 3 can compare against the clean baseline model."
    )

In [3]:
df = pd.read_csv(
    DATA_PATH,
    usecols=USECOLS,
    dtype={
        "comment_text": "string",
        "toxic": "float32",
        "black": "float32",
        "white": "float32",
    },
)

df = df.dropna(subset=["comment_text", "toxic"]).copy()
df["label"] = (df["toxic"] >= 0.5).astype(int)

train_df, eval_df = train_test_split(
    df,
    train_size=100_000,
    test_size=20_000,
    stratify=df["label"],
    random_state=SEED,
)

train_df = train_df.reset_index(drop=True)
eval_df = eval_df.reset_index(drop=True)

print(f"Train rows: {len(train_df):,}")
print(f"Eval rows:  {len(eval_df):,}")
print("Eval label distribution:")
print(eval_df["label"].value_counts(normalize=True).sort_index())

Train rows: 100,000
Eval rows:  20,000
Eval label distribution:
label
0    0.92005
1    0.07995
Name: proportion, dtype: float64


In [4]:
def tokenize_texts(tokenizer, texts):
    ds = Dataset.from_dict({"comment_text": list(texts)})

    def tok(batch):
        return tokenizer(batch["comment_text"], truncation=True, max_length=128)

    ds = ds.map(tok, batched=True, remove_columns=["comment_text"])
    return ds

def predict_toxic_prob(trainer, tokenizer, texts):
    ds = tokenize_texts(tokenizer, texts)
    out = trainer.predict(ds)
    logits = out.predictions
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
    preds = (probs >= 0.5).astype(int)
    return probs, preds

def fnr_from_labels(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return float(fn / (fn + tp)) if (fn + tp) > 0 else np.nan

def evaluate_predictions(y_true, probs, threshold=0.5):
    preds = (probs >= threshold).astype(int)
    return {
        "Accuracy": accuracy_score(y_true, preds),
        "F1_macro": f1_score(y_true, preds, average="macro"),
        "FNR": fnr_from_labels(y_true, preds),
    }

## Attack 1: Character-Level Evasion

In [5]:
clean_tokenizer = AutoTokenizer.from_pretrained(PART1_CHECKPOINT_DIR)
clean_model = AutoModelForSequenceClassification.from_pretrained(PART1_CHECKPOINT_DIR, num_labels=2)

clean_infer_args = TrainingArguments(
    output_dir="checkpoints/part3_clean_inference_tmp",
    per_device_eval_batch_size=64 if torch.cuda.is_available() else 8,
    report_to="none",
)

clean_trainer = Trainer(
    model=clean_model,
    args=clean_infer_args,
    data_collator=DataCollatorWithPadding(tokenizer=clean_tokenizer),
)

eval_probs_clean, eval_preds_clean = predict_toxic_prob(clean_trainer, clean_tokenizer, eval_df["comment_text"].tolist())

eligible_mask = (eval_preds_clean == 1) & (eval_probs_clean >= ATTACK1_CONFIDENCE_MIN)
eligible_idx = np.where(eligible_mask)[0]

print(f"Eligible comments (pred=1 and confidence>={ATTACK1_CONFIDENCE_MIN}): {len(eligible_idx):,}")

sample_n = min(ATTACK1_SAMPLE_SIZE, len(eligible_idx))
if sample_n == 0:
    raise ValueError("No eligible toxic predictions found for Attack 1. Try lowering ATTACK1_CONFIDENCE_MIN.")

rng = np.random.default_rng(SEED)
sample_idx = rng.choice(eligible_idx, size=sample_n, replace=False)

attack1_df = eval_df.loc[sample_idx, ["comment_text", "label"]].copy().reset_index(drop=True)
attack1_df["clean_prob"] = eval_probs_clean[sample_idx]
attack1_df["clean_pred"] = eval_preds_clean[sample_idx]

print(f"Attack 1 sample size used: {sample_n}")

Map: 100%|██████████| 20000/20000 [00:02<00:00, 8995.24 examples/s]


Eligible comments (pred=1 and confidence>=0.7): 1,167
Attack 1 sample size used: 500


In [6]:
ZERO_WIDTH = "\u200b"

TOXIC_STEMS = [
    "hate", "stupid", "idiot", "dumb", "kill", "ugly", "trash", "moron",
    "racist", "terror", "fuck", "shit", "bitch", "fool", "nasty", "disgust",
]

HOMOGLYPH_MAP = {
    "a": "а", "e": "е", "o": "о", "p": "р", "c": "с", "x": "х", "y": "у", "i": "і",
    "A": "А", "E": "Е", "O": "О", "P": "Р", "C": "С", "X": "Х", "Y": "У", "I": "І",
}

def is_toxic_looking(word):
    wl = word.lower()
    return any(stem in wl for stem in TOXIC_STEMS)

def insert_zero_width_chunks(word, rng_local):
    if len(word) < 4:
        return word
    chunks = []
    i = 0
    while i < len(word):
        step = int(rng_local.choice([2, 3]))
        chunks.append(word[i:i + step])
        i += step
    return ZERO_WIDTH.join(chunks)

def apply_zero_width_to_toxic_words(text, rng_local):
    def repl(match):
        w = match.group(0)
        if is_toxic_looking(w):
            return insert_zero_width_chunks(w, rng_local)
        return w
    return re.sub(r"[A-Za-z]+", repl, text)

def apply_homoglyphs(text):
    return "".join(HOMOGLYPH_MAP.get(ch, ch) for ch in text)

def duplicate_random_chars_in_words(text, rng_local, dup_rate=0.20):
    def repl(match):
        w = match.group(0)
        out = []
        for ch in w:
            out.append(ch)
            if rng_local.random() < dup_rate:
                out.append(ch)
        return "".join(out)
    return re.sub(r"[^\W\d_]+", repl, text, flags=re.UNICODE)

def perturb(text, seed_offset=0):
    # Deterministic per text+offset so results are reproducible.
    rng_local = np.random.default_rng(abs(hash((text, seed_offset))) % (2**32))

    t1 = apply_zero_width_to_toxic_words(text, rng_local)
    t2 = apply_homoglyphs(t1)
    t3 = duplicate_random_chars_in_words(t2, rng_local, dup_rate=0.20)
    return t3

In [9]:
attack1_df["perturbed_text"] = [perturb(t, seed_offset=i) for i, t in enumerate(attack1_df["comment_text"].tolist())]

perturbed_probs, perturbed_preds = predict_toxic_prob(
    clean_trainer,
    clean_tokenizer,
    attack1_df["perturbed_text"].tolist(),
)

attack1_df["perturbed_prob"] = perturbed_probs
attack1_df["perturbed_pred"] = perturbed_preds

flipped_to_nontoxic = ((attack1_df["clean_pred"] == 1) & (attack1_df["perturbed_pred"] == 0)).sum()
asr = flipped_to_nontoxic / len(attack1_df)

attack1_summary = pd.DataFrame([
    {
        "Sample_Size": len(attack1_df),
        "Eligible_Count": len(eligible_idx),
        "ASR": asr,
        "Avg_Confidence_Before": float(attack1_df["clean_prob"].mean()),
        "Avg_Confidence_After": float(attack1_df["perturbed_prob"].mean()),
        "Mean_Delta_Confidence": float((attack1_df["perturbed_prob"] - attack1_df["clean_prob"]).mean()),
    }
])

print("Attack 1 ASR Summary")
print(attack1_summary.round(4).to_string(index=False))

preview_cols = ["comment_text", "perturbed_text", "clean_prob", "perturbed_prob", "clean_pred", "perturbed_pred"]
print("\nAttack 1 preview (first 5 rows):")
print(attack1_df[preview_cols].head(5).to_string())

Map: 100%|██████████| 500/500 [00:00<00:00, 4061.30 examples/s]


Attack 1 ASR Summary
 Sample_Size  Eligible_Count   ASR  Avg_Confidence_Before  Avg_Confidence_After  Mean_Delta_Confidence
         500            1167 0.996                 0.9364                0.0055                -0.9308

Attack 1 preview (first 5 rows):
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   

## Attack 2: Label-Flipping Poisoning

In [10]:
poisoned_train_df = train_df[["comment_text", "label"]].copy()

flip_n = int(round(POISON_FRAC * len(poisoned_train_df)))
flip_rng = np.random.default_rng(SEED)
flip_idx = flip_rng.choice(poisoned_train_df.index.to_numpy(), size=flip_n, replace=False)

poisoned_train_df.loc[flip_idx, "label"] = 1 - poisoned_train_df.loc[flip_idx, "label"]

print(f"Total training rows: {len(poisoned_train_df):,}")
print(f"Flipped rows: {flip_n:,} ({POISON_FRAC:.1%})")

clean_train_df_for_ref = train_df[["comment_text", "label"]].copy()
clean_eval_df_for_ref = eval_df[["comment_text", "label"]].copy()

Total training rows: 100,000
Flipped rows: 5,000 (5.0%)


In [11]:
poison_tokenizer = AutoTokenizer.from_pretrained(MODEL_BASE)

clean_eval_hf = Dataset.from_pandas(clean_eval_df_for_ref, preserve_index=False)
poison_train_hf = Dataset.from_pandas(poisoned_train_df, preserve_index=False)

def poison_tok(batch):
    return poison_tokenizer(batch["comment_text"], truncation=True, max_length=128)

clean_eval_hf = clean_eval_hf.map(poison_tok, batched=True, remove_columns=["comment_text"])
poison_train_hf = poison_train_hf.map(poison_tok, batched=True, remove_columns=["comment_text"])

clean_eval_hf = clean_eval_hf.rename_column("label", "labels")
poison_train_hf = poison_train_hf.rename_column("label", "labels")

poison_model = AutoModelForSequenceClassification.from_pretrained(MODEL_BASE, num_labels=2)

poison_train_args = TrainingArguments(
    output_dir="checkpoints/part3_poisoned_distilbert",
    num_train_epochs=POISON_EPOCHS,
    learning_rate=2e-5,
    weight_decay=0.01,
    per_device_train_batch_size=16 if torch.cuda.is_available() else 4,
    per_device_eval_batch_size=32 if torch.cuda.is_available() else 8,
    gradient_accumulation_steps=1 if torch.cuda.is_available() else 4,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED,
)

poison_trainer = Trainer(
    model=poison_model,
    args=poison_train_args,
    train_dataset=poison_train_hf,
    eval_dataset=clean_eval_hf,
    data_collator=DataCollatorWithPadding(tokenizer=poison_tokenizer),
)

poison_train_out = poison_trainer.train()
print(poison_train_out)

poison_ckpt = os.path.join("checkpoints", "part3_poisoned_distilbert")
poison_trainer.save_model(poison_ckpt)
poison_tokenizer.save_pretrained(poison_ckpt)
print(f"Saved poisoned model checkpoint to: {poison_ckpt}")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 15465.15it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss
1,0.300428,0.167217
2,0.271627,0.163942
3,0.243639,0.190894


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.75it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=18750, training_loss=0.2643499908955892, metrics={'train_runtime': 456.6012, 'train_samples_per_second': 657.028, 'train_steps_per_second': 41.064, 'total_flos': 9844542409462080.0, 'train_loss': 0.2643499908955892, 'epoch': 3.0})


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.24it/s]

Saved poisoned model checkpoint to: checkpoints\part3_poisoned_distilbert


In [12]:
# Clean baseline metrics on eval set (using Part 1 clean model).
clean_metrics = evaluate_predictions(eval_df["label"].to_numpy(), eval_probs_clean, threshold=0.5)

# Poisoned model metrics on the same clean eval set.
poison_eval_out = poison_trainer.predict(clean_eval_hf)
poison_logits = poison_eval_out.predictions
poison_probs = torch.softmax(torch.tensor(poison_logits), dim=-1).numpy()[:, 1]
poison_metrics = evaluate_predictions(eval_df["label"].to_numpy(), poison_probs, threshold=0.5)

comparison = pd.DataFrame([
    {"Model": "Clean baseline (Part 1 checkpoint)", **clean_metrics},
    {"Model": "Poisoned retrain (5% flipped labels)", **poison_metrics},
])

delta_row = {
    "Model": "Delta (Poisoned - Clean)",
    "Accuracy": poison_metrics["Accuracy"] - clean_metrics["Accuracy"],
    "F1_macro": poison_metrics["F1_macro"] - clean_metrics["F1_macro"],
    "FNR": poison_metrics["FNR"] - clean_metrics["FNR"],
}
comparison = pd.concat([comparison, pd.DataFrame([delta_row])], ignore_index=True)

print("Attack 2: Clean vs Poisoned Metrics on the Same Clean Eval Set")
print(comparison.round(4).to_string(index=False))

Attack 2: Clean vs Poisoned Metrics on the Same Clean Eval Set
                               Model  Accuracy  F1_macro    FNR
  Clean baseline (Part 1 checkpoint)    0.9474    0.8051 0.4184
Poisoned retrain (5% flipped labels)    0.9468    0.7941 0.4653
            Delta (Poisoned - Clean)   -0.0005   -0.0110 0.0469


## Key Question Answer

Based on these results, the **more operationally dangerous attack for a live social platform is character-level evasion**, even though poisoning can be severe in theory.

### Evidence from this run
Attack 2 (5% label-flipping poisoning) changed clean-set performance as follows:
- Accuracy: 0.9474 -> 0.9468 (delta -0.0005)
- F1_macro: 0.8051 -> 0.7941 (delta -0.0110)
- FNR: 0.4184 -> 0.4653 (delta +0.0469)

This shows poisoning did hurt the model, especially by increasing false negatives (more toxic comments missed), but the overall degradation in this run is moderate rather than catastrophic.

### Threat-model reasoning
- **Evasion attack realism:** very high. Any user can obfuscate text at inference time with no access to internal systems.
- **Poisoning attack realism:** lower. It usually requires access to the training pipeline, labeling process, or data ingestion controls.

### Operational implication
For a real platform, prioritize defenses against evasion first because it is easier to execute at scale by ordinary adversarial users. At the same time, poisoning should still be treated as a high-impact internal security risk and mitigated with strict data governance (access control, provenance checks, and label-audit pipelines).